# FinDirector — LoRA Fine-Tuning on Colab

Fine-tune Qwen 2.5 7B Instruct with LoRA adapters on 696 synthetic financial queries.

**Setup:**
- Runtime: L4 GPU (24 GB VRAM)
- Estimated time: 30-90 minutes
- Cost: $0-5 in Colab compute units

**Prerequisites:**
- Colab Pro subscription with L4 GPU access
- Secrets configured: `HF_TOKEN`, `WANDB_API_KEY`
- W&B account with project access

**Pipeline this notebook runs:**
1. Verify GPU allocation
2. Clone the FinDirector repo
3. Install requirements
4. Load secrets from Colab Secrets manager
5. Run `scripts/train_directive_model.py`
6. Save trained adapter to Hugging Face Hub

## Step 1 — Verify GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone the FinDirector Repo

Get the latest training script and config from GitHub.

In [ ]:
# Clone the repo (public HTTPS clone, no auth needed for public repos)
# For private repos, we'd use a GitHub token via Colab Secrets
!git clone https://github.com/ShantanuBapat/findirector.git
%cd findirector
!git log -1 --oneline  # Show the latest commit for reference

## Step 3 — Install Requirements

Install all training dependencies. Colab has some pre-installed but we
force upgrade to match our pinned versions.

In [ ]:
# Install all requirements
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

## Step 4 — Load Secrets from Colab Secrets Manager

Access `HF_TOKEN` and `WANDB_API_KEY` without hardcoding them in the notebook.

In [ ]:
import os
from google.colab import userdata

# Load secrets into environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

# Verify (without printing the actual values)
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN not loaded correctly"
assert len(os.environ["WANDB_API_KEY"]) > 20, "WANDB_API_KEY not loaded correctly"

print("Secrets loaded successfully")
print(f"HF_TOKEN length: {len(os.environ['HF_TOKEN'])}")
print(f"WANDB_API_KEY length: {len(os.environ['WANDB_API_KEY'])}")

## Step 5 — Download Dataset from HF Hub

Pull the labeled splits from `AlHindi/findirector-splits` (our private dataset repo).

The training script expects splits at `data/synthetic/splits/`.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

# Download dataset files to local path expected by training script
splits_dir = Path("data/synthetic/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="AlHindi/findirector-splits",
    repo_type="dataset",
    local_dir=str(splits_dir),
    token=os.environ["HF_TOKEN"],
)

# Verify all 3 files present
for filename in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = splits_dir / filename
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"✓ {filename}: {size_kb:.1f} KB")
    else:
        print(f"✗ {filename}: MISSING")

## Step 6 — Run Training

Execute the training script. Progress logs stream to this notebook's output.

**Expected duration:** 30-90 minutes on L4 GPU.
**W&B dashboard:** https://wandb.ai/{your-username}/findirector-directive-model

In [ ]:
# Run training as a subprocess so output streams to the notebook
!python -m scripts.train_directive_model

## Step 7 — Save Trained Adapter to Hugging Face Hub

Push the LoRA adapter (small — ~200 MB) to a private HF Hub model repo
so it persists after this Colab session ends.

In [ ]:
from huggingface_hub import create_repo, upload_folder
from pathlib import Path

MODEL_REPO_ID = "AlHindi/findirector-directive-lora"
OUTPUT_DIR = Path("outputs/qwen-findirector-lora")

# Create private model repo
create_repo(
    repo_id=MODEL_REPO_ID,
    token=os.environ["HF_TOKEN"],
    repo_type="model",
    private=True,
    exist_ok=True,
)
print(f"Repo ready: {MODEL_REPO_ID}")

# Upload the trained adapter directory
if OUTPUT_DIR.exists():
    upload_folder(
        repo_id=MODEL_REPO_ID,
        folder_path=str(OUTPUT_DIR),
        repo_type="model",
        token=os.environ["HF_TOKEN"],
        commit_message="feat: initial LoRA adapter from Session 2.5 training",
    )
    print(f"Adapter pushed to https://huggingface.co/{MODEL_REPO_ID}")
else:
    print(f"WARNING: {OUTPUT_DIR} does not exist. Training may have failed.")

## Done!

Trained model available at: **https://huggingface.co/AlHindi/findirector-directive-lora**

**Next steps (Session 2.6):**
- Load the adapter for inference
- Evaluate against held-out test set (156 examples)
- Compare Qwen's predictions to Claude's labels
- Report per-code accuracy and confusion matrix